## Imports

In [57]:
import json
import math
import re
from datetime import datetime
from typing import Optional

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import boto3

## Configs

In [58]:
CURRENT_YEAR      = datetime.now().year
DECAY_RATE        = 0.10        # 10% reduction per year
DECAY_FLOOR       = 0.20        # minimum skill score
CONF_1_SOURCE     = 0.70        # single source confidence
CONF_2_SOURCES    = 0.85        # dual source confidence
CONF_3PLUS        = 1.00        # multi-source confidence
SENT_POSITIVE     = +0.15       # positive review boost
SENT_NEGATIVE     = -0.10       # negative review penalty
SOFT_SKILL_WEIGHT = 0.02        # per soft skill bonus
SOFT_SKILL_CAP    = 0.10        # max soft bonus total
NTH_MULTIPLIER    = 0.50        # nice-to-have multiplier
MAX_SKILL_WEIGHT  = 0.40        # max single skill weight
IC_SOFT_CAP       = 0.25        # IC role soft skill cap
MGR_SOFT_CAP      = 0.35        # managerial soft skill cap
SCORE_ALERT_THRESHOLD = 0.05    # notify if score changes >5%
BEDROCK_MODEL     = "arn:aws:bedrock:us-east-1:386275436225:inference-profile/global.anthropic.claude-haiku-4-5-20251001-v1:0"
BEDROCK_REGION    = "us-east-1" # change if your region differs

## Connection to BedRock

In [59]:
def call_bedrock(prompt: str, system_prompt: str="", max_tokens = 1000) -> str:
    client = boto3.client("bedrock-runtime", region_name=BEDROCK_REGION)
    
    messages = [{"role": "user", "content": prompt}]
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "messages": messages
    }
    
    if system_prompt:
        body["system"] = system_prompt
    
    response = client.invoke_model(
        modelId = BEDROCK_MODEL,
        body = json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )
    
    result = json.loads(response["body"].read())
    return result["content"][0]["text"]

def call_bedrock_json(prompt: str, system_prompt: str="", max_tokens: int=1000) -> dict:
    raw = call_bedrock(prompt, system_prompt, max_tokens)
    
    cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
    
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"Json Parse error: {e}")
        print(f"Raw Response: {raw[:500]}")
        raise

def call_bedrock_json_safe(prompt, system_prompt="", max_tokens=1000, retries=3):
    for attempt in range(retries):
        try:
            return call_bedrock_json(prompt, system_prompt, max_tokens)
        except (json.JSONDecodeError, ValueError, KeyError) as e:
            print(f"Attempt {attempt+1} failed: {e}, retrying...")
            system_prompt+=f"\n Received an Error while Parsing JSON in your previous response. please try again.\nHere are the details about the error: \n{e}"
    raise ValueError(f"Failed to get valid JSON after {retries} attempts.")


# ── Test the connection ───────────────────────────────────
print("Testing Bedrock connection...")

test_response = call_bedrock_json_safe(
    prompt="""Return a JSON object with exactly these fields:
    {
        "status": "connected",
        "model": "claude-3-haiku",
        "message": "IntelliMove pipeline ready"
    }
    Return ONLY the JSON. No explanation."""
)

print(f"Bedrock connection successful!")
print(f"   Status:  {test_response.get('status')}")
print(f"   Model:   {test_response.get('model')}")
print(f"   Message: {test_response.get('message')}")

Testing Bedrock connection...
Bedrock connection successful!
   Status:  connected
   Model:   claude-3-haiku
   Message: IntelliMove pipeline ready


## Data Loading

In [60]:
JOB_DESCRIPTIONS = {}
EMPLOYEES = {}
SKILLS_DICT = {}
SKILL_GROUPS = {}

with open("data/job_descriptions.json", "r") as f:
    JOB_DESCRIPTIONS = json.load(f)
    
with open("data/employees.json", "r") as f:
    EMPLOYEES = json.load(f)
    
with open("data/skills_dict.json", "r") as f:
    SKILLS_DICT = json.load(f)

with open("data/skill_groups.json", "r") as f:
    SKILL_GROUPS = json.load(f)
 


# ── Print summary ─────────────────────────────────────────
print("✅ Cell 3 complete — synthetic data loaded")
print(f"\n   Job Descriptions: {len(JOB_DESCRIPTIONS)}")
for jd_id, jd in JOB_DESCRIPTIONS.items():
    print(f"   {jd_id}: {jd['title']} ({jd['department']})")

print(f"\n   Employees: {len(EMPLOYEES)}")
for emp_id, emp in EMPLOYEES.items():
    docs = len(emp['documents'])
    print(f"   {emp_id}: {emp['name']} — {emp['current_role']} ({docs} sources)")


print(f"   Normalization dictionary: {len(SKILLS_DICT)} entries (will grow)")
print(f"   Skills Group: {len(SKILL_GROUPS)} entries (will Grow)")

✅ Cell 3 complete — synthetic data loaded

   Job Descriptions: 3
   JD001: Senior Data Engineer (Healthcare Analytics)
   JD002: ML Engineer (AI Team)
   JD003: Healthcare IT Manager (Clinical Systems)

   Employees: 6
   E001: Alex Rivera — Data Engineer (3 sources)
   E002: Maria Chen — Data Analyst (2 sources)
   E003: James Patel — Software Engineer (3 sources)
   E004: Sarah Kim — ML Research Engineer (3 sources)
   E005: David Osei — Healthcare IT Specialist (3 sources)
   E006: Priya Nair — IT Project Manager (2 sources)
   Normalization dictionary: 0 entries (will grow)
   Skills Group: 0 entries (will Grow)


## Skill Normalization

In [61]:
def get_skill_group(canonical_skill: str) -> Optional[str]:
    return SKILL_GROUPS.get(canonical_skill)

def get_existing_groups() -> list:
    return list(set(SKILL_GROUPS.values()))

def save_skills_dict():
    with open("data/skills_dict.json", "w") as f:
        json.dump(SKILLS_DICT, f, indent=2)

def save_skill_groups():
    with open("data/skill_groups.json", "w") as f:
        json.dump(SKILL_GROUPS, f, indent=2)

def build_normalization_prompt(skill_name: str) -> str:
    existing_groups = get_existing_groups()

    if existing_groups:
        groups_str = "\n".join(f"  - {g}" for g in sorted(existing_groups))
        groups_instruction = f"""Existing skill groups (reuse if appropriate):
{groups_str}

Either assign the skill to one of the existing groups above,
or create a new group name if none fits."""
    else:
        groups_instruction = """No groups exist yet.
Create an appropriate group name for this skill.
Examples of good group names: ML Frameworks, Cloud Platforms,
Data Processing, Programming Languages, Healthcare Standards,
Project Management, Soft Skills."""

    return f"""You are a skill normalization engine.

Your job is to canonicalize a raw skill name AND assign it to a skill group.

Raw skill: "{skill_name}"

Canonicalization rules:
1. Return the most widely recognized canonical form
2. Fix abbreviations: "ML" → "Machine Learning", "TF" → "TensorFlow"
3. Fix casing: "apache spark" → "Apache Spark"
4. Fix verbose names: "Python programming" → "Python"
5. If already canonical, return as-is

Skill group rules:
1. Group skills that are transferable or closely related
2. Examples:
   - TensorFlow, PyTorch, Keras → "ML Frameworks"
   - AWS, Azure, GCP → "Cloud Platforms"
   - Apache Spark, Hadoop, Flink → "Data Processing"
   - Python, Java, Scala → "Programming Languages"
   - HIPAA, HL7, FHIR → "Healthcare Standards"
   - Leadership, Communication, Teamwork → "Soft Skills"
3. Skills in the same group get partial credit during matching

{groups_instruction}

Return ONLY this exact JSON structure. No explanation. No markdown:
{{"canonical_skill_name": "canonical name here","group": "group name here"}}"""


def normalize_via_llm(skill_name: str) -> tuple:
    """
    Uses LLM when dictionary lookup misses.
    """
    prompt = build_normalization_prompt(skill_name=skill_name)
    result = call_bedrock_json_safe(prompt=prompt, max_tokens=100)
    
    canonical = result.get("canonical_skill_name", "").strip()
    group = result["group"].strip()
    
    return canonical, group

def normalize(skill_name: str) -> str:
    """
    Main normalization function.
    1. Check SKILLS_DICT
    2. LLM fallback if not found.
    3. Update SKILLS_DICT + SKILL_GROUPS
    4. Persist both JSON files
    Returns canonical skill name.
    """
    
    key = skill_name.lower().strip()
    
    if key in SKILLS_DICT:
        return SKILLS_DICT[key]
    
    canonical, group = normalize_via_llm(skill_name)
    
    SKILLS_DICT[key] = canonical
    SKILL_GROUPS[canonical] = group
    
    save_skills_dict()
    save_skill_groups()

    return canonical

def normalize_skills_list(skills: list, name_key: str = "skill") -> list:
    print(f" Normalizing {len(skills)} skills...")
    
    for skill in skills:
        raw_name = skill[name_key]
        canonical = normalize(raw_name)
        if canonical != raw_name:
            print(f" '{raw_name}' -> '{canonical}'")
        skill[name_key] = canonical
    return skills


test_skills = [
    # Should group together as ML Frameworks
    "TensorFlow",
    "PyTorch",
    "Keras",
    # Should group together as Data Processing
    "Apache Spark",
    "Spark",          # dict hit after Apache Spark
    "Hadoop",
    # Should group together as Healthcare Standards
    "HIPAA",
    "HL7",
    "FHIR",
    # Soft skills
    "team player",
    "Leadership",
]

print("\nTesting normalizer:")
print(f"  {'Raw Skill':<25} {'Canonical':<25} {'Group':<22} {'Source'}")
print(f"  {'-'*75}")

for raw in test_skills:
    key       = raw.lower().strip()
    was_cached = key in SKILLS_DICT
    canonical  = normalize(raw)
    group      = get_skill_group(canonical) or "—"
    source     = "dict" if was_cached else "LLM"
    print(f"  {raw:<25} {canonical:<25} {group:<22} {source}")


# Print groups formed
print(f"\n  Groups formed:")
groups_formed = {}
for skill, group in SKILL_GROUPS.items():
    if group not in groups_formed:
        groups_formed[group] = []
    groups_formed[group].append(skill)

for group, skills in sorted(groups_formed.items()):
    print(f"  [{group}]")
    for s in skills:
        print(f"    - {s}")



Testing normalizer:
  Raw Skill                 Canonical                 Group                  Source
  ---------------------------------------------------------------------------
  TensorFlow                TensorFlow                ML Frameworks          LLM
  PyTorch                   PyTorch                   ML Frameworks          LLM
  Keras                     Keras                     ML Frameworks          LLM
  Apache Spark              Apache Spark              Data Processing        LLM
  Spark                     Apache Spark              Data Processing        LLM
  Hadoop                    Hadoop                    Data Processing        LLM
  HIPAA                     HIPAA                     Healthcare Standards   LLM
  HL7                       HL7                       Healthcare Standards   LLM
  FHIR                      FHIR                      Healthcare Standards   LLM
  team player               Teamwork                  Soft Skills            LLM
  Leade

## JD Processor

In [ ]:
def build_jd_prompt(jd: dict) -> str:
    return python# ============================================================
# CELL 5 — JD Processor (Complete Pipeline)
# LLM extract → multiplier → normalize → deduplicate → renormalize
# ============================================================

def build_jd_prompt(jd: dict) -> str:
    """Formats JD into structured prompt for LLM."""
    return f"""You are an expert HR skill analyst.

Analyze the following job description and extract all skills with weights.

JOB TITLE: {jd['title']}

ABOUT THE ROLE:
{jd['about']}

RESPONSIBILITIES:
{jd['responsibilities']}

REQUIREMENTS:
{jd['requirements']}

NICE TO HAVE:
{jd['nice_to_have']}

WEIGHTING RUBRIC — follow strictly:

0. Classify role as "individual_contributor" or "managerial".
   Signals for MANAGERIAL: "lead a team", "manage X people",
   "oversee a team", "people manager", "director", "VP".
   NOTE: "mentor" or "collaborate" alone does NOT make a role managerial.
   Individual contributors who mentor are still IC roles.

1. Identify ALL skills mentioned across all sections.
   Skills include: technical tools, programming languages,
   domain knowledge, and soft skills.

2. Assign a raw weight to each skill based on:
   a. Frequency  — mentioned more than once = higher importance
   b. Position   — listed earlier = more important
   c. Role type  — technical > soft for IC; leadership > technical for managerial
   d. Explicitness — "must have" language = higher weight

3. Weights must sum to exactly 1.0 across ALL skills.
4. No single skill should exceed 0.40.
5. Soft skills collectively must not exceed the role-type cap.
6. Tag each skill with source section: "required" or "nice_to_have".
7. Include role_type in output.

Return ONLY this exact JSON. No explanation. No markdown:
{{"role_type": "individual_contributor or managerial", "skills": [{{"skill": "skill name", "weight": 0.00, "section": "required or nice_to_have"}}]}}"""


def apply_section_multiplier(skills: list) -> list:
    """
    Applies 1.0 multiplier to required skills, 0.5 multiplier to nice_to_have_skills.
    why? because we don't want nice_to_have skills weight exceed the required skills weight.
    """
    
    for skill in skills:
        if skill["section"] == "nice_to_have":
            skill["weight"] = round(skill["weight"] * NTH_MULTIPLIER, 6)
        return skills
    

def deduplicate_skills(skills: list) -> list:
    """
    After normalization, merges duplicate canonical skill names.
    Sums weights (capped at MAX_SKILL_WEIGHT).
    Takes 'required' if any duplicate was required.
    """
    merged = {}
    for skill in skills:
        name = skill["skill"].lower().strip()
        if name in merged:
            merged[name]["weight"] = min(merged[name]["weight"]+skill["weight"], MAX_SKILL_WEIGHT)
            
            if skill["section"] == "required":
                merged[name]["section"]="required"
        else:
            merged[name] = {
                "skill": skill["skill"],
                "weight": skill["weight"],
                "section": skill["section"]
            }
    
    return list(merged.values())

def renormalize_weights(skills: list) -> list:
    """
    Re-normalizes all skill weights so they sum to 1.0.
    Called after multiplier and deduplication.
    """
    
    total = sum(s["weight"] for s in skills)
    if total == 0:
        return skills
    for skill in skills:
        skill["weight"] = round(skill["weight"]/total, 4)
    return skills


def process_jd(jd: dict) -> dict:
    """
    1. LLM extracts skills + weights
    2. Apply section Multiplier
    3. Normalize skill names
    4. Deduplicate
    5. Re-normalize weights to ~1.0
    """
    
    print(f"\n\n  Processing: {jd["title"]}...")
    # Step 1
    prompt = build_jd_prompt(jd)
    raw = call_bedrock_json_safe(prompt, max_tokens=1000)
    
    role_type = raw.get("role_type", "individual_contributor")
    skills    = raw.get("skills", [])
    
    print(f"    Role type:            {role_type}")
    print(f"    Raw Skills:           {len(skills)} extracted")
    
    soft_cap = MGR_SOFT_CAP if role_type == "managerial" else IC_SOFT_CAP
    print(f"    Soft skill cap: {soft_cap}")
    
    
    #step 2
    skills = apply_section_multiplier(skills)
    print(F"    Multiplier: applied (NtH * {NTH_MULTIPLIER})")
    
    #step 3
    skills = normalize_skills_list(skills, name_key="skill")
    
    #step 4
    skills = deduplicate_skills(skills)
    
    #step 5
    skills = renormalize_weights(skills)
    
    # weights validation
    total = sum(s["weight"] for s in skills)
    print(f"    Weights sum:            {total:.4f} {'All good' if abs(total - 1.0) < 0.01 else 'Weights not sum to 1'}")
    
    return {
        "jd_id":     jd["id"],
        "title":     jd["title"],
        "role_type": role_type,
        "soft_cap":  soft_cap,
        "skills":    skills
    }
    

def show_jd_result(result: dict):
    print(f"\n{'='*60}")
    print(f"  {result['title']}  ({result['jd_id']})")
    print(f"  Role type: {result['role_type']}  |  Soft cap: {result['soft_cap']}")
    print(f"{'='*60}")
    print(f"  {'Skill':<28} {'Section':<15} {'Weight':>8}  {'Group'}")
    print(f"  {'-'*65}")
    for s in sorted(result["skills"], key=lambda x: x["weight"], reverse=True):
        group = get_skill_group(s["skill"]) or "—"
        print(f"  {s['skill']:<28} {s['section']:<15} {s['weight']:>8.4f}  {group}")
    print(f"  {'-'*65}")
    total = sum(s["weight"] for s in result["skills"])
    print(f"  {'TOTAL':<28} {'':<15} {total:>8.4f}")


JD_RESULTS = {}

for jd_id, jd in JOB_DESCRIPTIONS.items():
    result = process_jd(jd)
    JD_RESULTS[jd_id] = result
    show_jd_result(result)


  Processing: Senior Data Engineer...
  Role type:            individual_contributor
  Raw Skills:           9 extracted
    Soft skill cap: 0.25
    Multiplier: applied (NtH * 0.5)
 Normalizing 9 skills...
 'Data Quality and Reliability' -> 'Data Quality'
 'dbt (Data Build Tool)' -> 'dbt'
    Weights sum:            1.0000 All good

  Senior Data Engineer  (JD001)
  Role type: individual_contributor  |  Soft cap: 0.25
  Skill                        Section           Weight  Group
  -----------------------------------------------------------------
  Python                       required          0.1800  Programming Languages
  Apache Spark                 required          0.1800  Data Processing
  SQL                          required          0.1600  Query Languages
  Data Pipeline Design and Maintenance required          0.1600  Data Processing
  Leadership and Team Collaboration required          0.1200  Soft Skills
  Data Quality                 required          0.0800  Data Man